In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!pip -q install -U "datasets>=3.0.0" tqdm

import ast
import hashlib
import json
import math
import os
import random
from collections import defaultdict
from functools import lru_cache
from itertools import combinations, product

import numpy as np
from datasets import load_dataset

ROOT = "/content/drive/MyDrive/Test"
NS = (2, 3, 4)
PEOPLE = ("Eyal", "Shahar", "Noa", "Shelley")
TRUTH = {"k", "t", "truth", "knight", "telling-truth", "truth-teller"}
LIE = {"n", "l", "lie", "liar", "knave", "lying", "not-knight"}
IFF = {"<->", "<=>"}

os.environ["TOKENIZERS_PARALLELISM"] = "false"


def dirs(*xs):
    for x in xs:
        os.makedirs(x, exist_ok=True)


def run_dirs(name, res=None):
    a = os.path.join(ROOT, name)
    b = os.path.join(a, res or f"{name.lower()}_results")
    dirs(a, b)
    return a, b


def b(x):
    return "True" if bool(x) else "False"


def lit(x):
    return ast.literal_eval(x) if isinstance(x, str) else x


def tree(x):
    return tuple(tree(y) for y in x) if isinstance(x, (list, tuple)) else x


def clean(x):
    return x.strip().lower().replace("_", "-").replace(" ", "-") if isinstance(x, str) else None


def seed(x):
    return int(hashlib.md5(x.encode()).hexdigest()[:8], 16)


def uid(x):
    x = x if isinstance(x, str) else repr(x)
    return hashlib.md5(x.encode()).hexdigest()


def vars_n(n):
    return tuple(chr(ord("a") + i) for i in range(n))


def people_n(n):
    return PEOPLE[:n] if n <= len(PEOPLE) else tuple(f"Person {i + 1}" for i in range(n))


def save_json(path, x):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(x, f, indent=2, ensure_ascii=False)


def save_jsonl(path, rows):
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]


def split_n(name):
    return int(name[:-3])


def dist(a, b_):
    return sum(x != y for x, y in zip(a, b_))


@lru_cache(maxsize=None)
def assigns(n):
    return tuple(tuple(bool((bits >> i) & 1) for i in range(n)) for bits in range(1 << n))


def wchoice(items, weights, rng):
    return rng.choice(items) if not any(weights) else rng.choices(items, weights=weights, k=1)[0]


def rate(rows):
    return float(np.mean([int(r["label"]) for r in rows])) if rows else 0.0


def rname(i):
    return f"rule_{i + 1}"


def neg_atom(x):
    if x[0] == "const":
        return ("const", not x[1])
    if x[0] == "var":
        return ("not", x[1])
    if x[0] == "not":
        return ("var", x[1])
    raise ValueError(x)


def parse_atom(x):
    x = tree(x)

    if isinstance(x, bool):
        return ("const", bool(x))
    if isinstance(x, int):
        return ("var", x)

    tag = clean(x[0])
    arg = x[1]

    if tag in TRUTH:
        return ("var", arg)
    if tag in LIE:
        return ("not", arg)
    if tag == "not":
        return neg_atom(parse_atom(arg))

    raise ValueError(x)


def parse_expr(x):
    x = tree(x)

    if isinstance(x, (bool, int)) or len(x) == 2:
        return ("atom", parse_atom(x))

    tag = clean(x[0])
    a, c = x[1], x[2]

    if tag == "and":
        return ("and", parse_atom(a), parse_atom(c))
    if tag == "or":
        return ("or", parse_atom(a), parse_atom(c))
    if tag in IFF:
        return ("iff", parse_atom(a), parse_atom(c))
    if tag == "->":
        return ("or", neg_atom(parse_atom(a)), parse_atom(c))

    raise ValueError(x)


def val_atom(x, a):
    if x[0] == "const":
        return bool(x[1])
    if x[0] == "var":
        return bool(a[x[1]])
    if x[0] == "not":
        return not bool(a[x[1]])
    raise ValueError(x)


def val_expr(x, a):
    if x[0] == "atom":
        return val_atom(x[1], a)
    if x[0] == "and":
        return val_atom(x[1], a) and val_atom(x[2], a)
    if x[0] == "or":
        return val_atom(x[1], a) or val_atom(x[2], a)
    if x[0] == "iff":
        return val_atom(x[1], a) == val_atom(x[2], a)
    raise ValueError(x)


def ok(st, a):
    return all(a[i] == val_expr(st[i], a) for i in range(len(st)))


def good_count(st, a):
    return sum(a[i] == val_expr(st[i], a) for i in range(len(st)))


def render_atom(x, names, lang):
    if x[0] == "const":
        return b(x[1])
    name = names[x[1]]
    if lang == "py":
        return name if x[0] == "var" else f"(not {name})"
    return f"{name} is truthful" if x[0] == "var" else f"{name} is lying"


def render_expr(x, names, lang):
    if x[0] == "atom":
        return render_atom(x[1], names, lang)

    left = render_atom(x[1], names, lang)
    right = render_atom(x[2], names, lang)

    if lang == "py":
        if x[0] == "and":
            return f"{left} and {right}"
        if x[0] == "or":
            return f"{left} or {right}"
        return f"{left} == {right}"

    if x[0] == "and":
        return f"both of the following are True: {left}; {right}"
    if x[0] == "or":
        return f"at least one of the following is True: {left}; {right}"
    return f"{left} exactly when {right}"


def prompt_py(st, a, vs):
    lines = [f"def check({', '.join(vs)}) -> bool:", ""]
    lines += [f"    {rname(i)} = {render_expr(x, vs, 'py')}" for i, x in enumerate(st)]
    lines += ["", "    return " + " and ".join(f"{v} == {rname(i)}" for i, v in enumerate(vs)), "", "Input:"]
    lines += [f"{v} = {b(x)}" for v, x in zip(vs, a)]
    lines += ["", "Output:"]
    return "\n".join(lines)


def prompt_en(st, a, vs, ps=None):
    ps = ps or people_n(len(vs))
    lines = [f"Given labels for {', '.join(ps)}, determine whether the assignment is True or False.", ""]
    lines += [f"{p} says: {render_expr(x, ps, 'en')}" for p, x in zip(ps, st)]
    lines += ["", "Determine whether all of the following hold:"]
    lines += [f"{p} is labeled truthful exactly when {p}'s statement is truthful." for p in ps]
    lines += ["", "Input:"]
    lines += [f"{p} is {'truthful' if x else 'lying'}." for p, x in zip(ps, a)]
    lines += ["", "Output:"]
    return "\n".join(lines)


def clean_gpu():
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
DATASET_ID = "K-and-K/knights-and-knaves"

PREP, PREP_RES = run_dirs("PreparedData", "prepared_data_results")

VAL_FRAC = 0.10
FALSE_SCALE = 1.0
MIN_TRUE = 4
MAX_TRUE = 24
N_MULT = {2: 0.90, 3: 1.15, 4: 1.50}
IFF_MULT = 1.25

PHASES = [
    ("main", 1, 1.0, None),
    ("finish", 2, 0.10, lambda p: p["n"] == 4 or p["has_iff"]),
]

P_TRUE_NEW = 0.65
P_FALSE_NEW = 0.70
D_BIAS = 0.20
P_BIAS = 0.15

USE_RETURN = False
RULE_MODES = [("local_check", "local_check")]


def split_by_label(st):
    all_a = assigns(len(st))
    true_a = [a for a in all_a if ok(st, a)]
    true_set = set(true_a)
    false_a = [a for a in all_a if a not in true_set]
    return true_a, true_set, false_a


def puzzles(cfg):
    out = {}
    ds = load_dataset(DATASET_ID, cfg)

    for split, rows in ds.items():
        n = split_n(split)
        if n not in NS:
            continue

        for row in rows:
            st = tuple(parse_expr(x) for x in tree(lit(row["statements"])))
            true_a, true_set, false_a = split_by_label(st)

            if len(true_a) == 0 or len(true_a) == (1 << n):
                continue

            k = uid(st)
            if k in out:
                continue

            out[k] = {
                "uid": k,
                "n": n,
                "var_names": vars_n(n),
                "person_names": people_n(n),
                "statements": st,
                "solutions": list(true_a),
                "solution_set": true_set,
                "false_assignments": false_a,
                "has_iff": any(x[0] == "iff" for x in st),
                "false_rule_match_count": {a: good_count(st, a) for a in false_a},
            }

    return list(out.values())


def example(p, a, split):
    label = a in p["solution_set"]
    return {
        "uid": p["uid"],
        "split": split,
        "n": p["n"],
        "assignment": {v: bool(x) for v, x in zip(p["var_names"], a)},
        "label": bool(label),
        "english_text": prompt_en(p["statements"], a, p["var_names"], p["person_names"]),
        "python_text": prompt_py(p["statements"], a, p["var_names"]),
    }


def split_train_val(ps):
    groups = defaultdict(list)
    for p in ps:
        groups[(p["n"], int(p["has_iff"]))].append(p)

    train, val = [], []
    for key, group in groups.items():
        group = sorted(group, key=lambda p: p["uid"])
        rng = random.Random(seed(f"val-split-{key}"))
        rng.shuffle(group)
        k = 0 if len(group) <= 1 else min(len(group) - 1, max(1, int(round(len(group) * VAL_FRAC))))
        val += group[:k]
        train += group[k:]

    return train, val


def draw_count(pool, p, scale):
    x = max(MIN_TRUE, int(math.ceil(FALSE_SCALE * len(pool))))
    if MAX_TRUE is not None:
        x = min(x, MAX_TRUE)

    m = N_MULT.get(p["n"], 1.0)
    if p["has_iff"]:
        m *= IFF_MULT

    x = max(1, int(round(x * m * scale)))
    if MAX_TRUE is not None:
        x = min(x, max(1, int(math.ceil(MAX_TRUE * max(1.0, m * scale)))))
    return x


def candidate_pool(pool, use, rng, p_new, last):
    new = [x for x in pool if use[x] == 0]
    old = [x for x in pool if use[x] > 0]
    xs = new if new and old and rng.random() < p_new else (old if new and old else new or old or pool)

    if last is not None and len(xs) > 1:
        ys = [x for x in xs if x != last]
        xs = ys or xs

    return xs


def pick_true(pool, use, rng, last):
    xs = candidate_pool(pool, use, rng, P_TRUE_NEW, last)
    x = wchoice(xs, [1.0 / (1.0 + use[a]) for a in xs], rng)
    use[x] += 1
    return x


def pick_false(pool, use, bucket_use, true_a, rng, last, p):
    xs = candidate_pool(pool, use, rng, P_FALSE_NEW, last)
    weights = []

    for a in xs:
        d = dist(a, true_a)
        u = 1.0 / (1.0 + use[a])
        v = 1.0 / (1.0 + bucket_use[d])
        close = 1.0 + D_BIAS * ((p["n"] - d) / max(1, p["n"]))
        plaus = p["false_rule_match_count"][a] / max(1, p["n"])
        weights.append(u * v * close * (1.0 + P_BIAS * plaus))

    x = wchoice(xs, weights, rng)
    d = dist(x, true_a)
    use[x] += 1
    bucket_use[d] += 1
    return x, d


def phase_rows(ps, split, false_per_true, scale, keep=None):
    rows, cov = [], []

    for p in ps:
        if keep is not None and not keep(p):
            continue

        true_pool = list(p["solutions"])
        false_pool = list(p["false_assignments"])
        n_true = draw_count(false_pool, p, scale)

        rng_t = random.Random(seed(f"{split}-true-{p['uid']}"))
        rng_f = random.Random(seed(f"{split}-false-{p['uid']}"))

        use_t = {a: 0 for a in true_pool}
        use_f = {a: 0 for a in false_pool}
        use_d = defaultdict(int)
        hist_d = defaultdict(int)

        last_t = None
        last_f = None

        for _ in range(n_true):
            a_t = pick_true(true_pool, use_t, rng_t, last_t)
            last_t = a_t
            rows.append(example(p, a_t, split))

            for _ in range(false_per_true):
                a_f, d = pick_false(false_pool, use_f, use_d, a_t, rng_f, last_f, p)
                hist_d[d] += 1
                last_f = a_f
                rows.append(example(p, a_f, split))

        cov.append({
            "uid": p["uid"],
            "split": split,
            "n": p["n"],
            "has_iff": bool(p["has_iff"]),
            "false_per_true": int(false_per_true),
            "phase_scale": float(scale),
            "num_true_available": len(true_pool),
            "num_false_available": len(false_pool),
            "true_draws": int(n_true),
            "num_true_used_total": int(sum(use_t.values())),
            "num_false_used_total": int(sum(use_f.values())),
            "num_true_used_unique": int(sum(x > 0 for x in use_t.values())),
            "num_false_used_unique": int(sum(x > 0 for x in use_f.values())),
            "true_full_coverage_reached": sum(x > 0 for x in use_t.values()) == len(true_pool),
            "false_full_coverage_reached": sum(x > 0 for x in use_f.values()) == len(false_pool),
            "true_usage_counts": {str(k): int(v) for k, v in sorted(use_t.items(), key=lambda kv: kv[0])},
            "false_usage_counts": {str(k): int(v) for k, v in sorted(use_f.items(), key=lambda kv: kv[0])},
            "false_distance_hist": {str(k): int(v) for k, v in sorted(hist_d.items())},
        })

    random.Random(seed("shuffle-" + split)).shuffle(rows)
    return rows, cov


def train_phases(ps):
    out = {}
    for name, fpt, scale, keep in PHASES:
        rows, cov = phase_rows(ps, f"train_{name}", fpt, scale, keep)
        out[name] = {"rows": rows, "coverage": cov}
    return out


def full_rows(ps, split):
    return [example(p, a, split) for p in ps for a in assigns(p["n"])]


def full_stats(ps):
    n = sum(1 << p["n"] for p in ps)
    t = sum(len(p["solutions"]) for p in ps)
    return {"num_examples": int(n), "positive_rate": float(t / n) if n else 0.0}


def project(rows, key):
    return [
        {
            "text": r[key],
            "label": r["label"],
            "uid": r["uid"],
            "assignment": r["assignment"],
            "n": r["n"],
            "split": r["split"],
        }
        for r in rows
    ]


def cov_summary(rows):
    def avg(k):
        return float(np.mean([r[k] for r in rows])) if rows else 0.0

    return {
        "num_puzzles": len(rows),
        "avg_true_draws": avg("true_draws"),
        "avg_false_per_true": avg("false_per_true"),
        "avg_true_available": avg("num_true_available"),
        "avg_false_available": avg("num_false_available"),
        "avg_true_unique_used": avg("num_true_used_unique"),
        "avg_false_unique_used": avg("num_false_used_unique"),
        "all_true_full_coverage": bool(all(r["true_full_coverage_reached"] for r in rows)) if rows else True,
        "all_false_full_coverage": bool(all(r["false_full_coverage_reached"] for r in rows)) if rows else True,
    }


def style_prompt(vs, body, a):
    lines = [f"def check({', '.join(vs)}) -> bool:", ""]
    for line in body:
        if line == "":
            lines.append("")
        elif line.startswith("check = "):
            lines.append("    return " + line[len("check = "):])
        else:
            lines.append("    " + line)
    lines += ["", "Input:"]
    lines += [f"{v} = {b(x)}" for v, x in zip(vs, a)]
    lines += ["", "Output:"]
    return "\n".join(lines)


def add_row(rows, family, n, vs, body, a, label, meta=None):
    text = style_prompt(vs, body, a)
    row = {
        "uid": uid(text + "|||" + b(label)),
        "family": family,
        "n": n,
        "assignment": {v: bool(x) for v, x in zip(vs, a)},
        "text": text,
        "label": bool(label),
    }
    if meta:
        row.update(meta)
    rows.append(row)


def add_all(rows, family, n, vs, body, fn, meta=None):
    for a in assigns(n):
        add_row(rows, family, n, vs, body, a, fn(a), meta)


def dedup(rows):
    seen, out = set(), []
    for r in rows:
        k = (r["text"], bool(r["label"]))
        if k not in seen:
            seen.add(k)
            out.append(r)
    return out


def signed_atoms(n):
    for i in range(n):
        yield ("var", i)
        yield ("not", i)


def atom_suffix(x):
    return f"{x[0]}_{x[1]}"


def atom_meta(prefix, x):
    return {
        f"{prefix}_kind": x[0],
        f"{prefix}_var": x[1],
        f"{prefix}_signed_atom": repr(x),
    }


def expr_specs(n):
    for atom in signed_atoms(n):
        kind, i = atom
        expr = ("atom", atom)
        yield ("var" if kind == "var" else "not"), expr, {
            "expr": repr(expr),
            "source_kind": kind,
            "source_var": i,
            "source_signed_atom": repr(atom),
            "expr_arity": 1,
            "expr_tag": "atom",
        }

    for left, right in product(signed_atoms(n), repeat=2):
        for name, tag in (("and", "and"), ("or", "or"), ("eq", "iff")):
            expr = (tag, left, right)
            yield f"{name}_{atom_suffix(left)}__{atom_suffix(right)}", expr, {
                "expr": repr(expr),
                "expr_arity": 2,
                "expr_tag": tag,
                **atom_meta("left", left),
                **atom_meta("right", right),
            }


def add_rule(rows, n, vs, family, target, expr, mode, meta=None):
    rr = rname(target)
    text = render_expr(expr, vs, "py")

    if mode == "rule_return":
        body = [f"{rr} = {text}", "", f"check = {rr}"]
        fn = lambda a, expr=expr: val_expr(expr, a)
    else:
        body = [f"{rr} = {text}", "", f"check = ({vs[target]} == {rr})"]
        fn = lambda a, target=target, expr=expr: bool(a[target]) == bool(val_expr(expr, a))

    info = {"target_rule": rr, "target_idx": target, "family_mode": mode, **(meta or {})}
    info.setdefault("rule_expr", repr(expr))
    add_all(rows, family, n, vs, body, fn, info)


def add_multi(rows, n, vs, family, targets, sources):
    body = [f"{rname(i)} = {vs[src]}" for i, src in enumerate(sources)]
    body += ["", "check = " + " and ".join(f"({vs[t]} == {rname(i)})" for i, t in enumerate(targets))]
    add_all(
        rows, family, n, vs, body,
        lambda a, targets=targets, sources=sources: all(bool(a[t]) == bool(a[s]) for t, s in zip(targets, sources)),
        {
            "target_vars": [vs[i] for i in targets],
            "source_vars": [vs[i] for i in sources],
            "num_checks": len(targets),
            "has_repeated_source": len(set(sources)) < len(sources),
        },
    )


def atom_rows_n(n):
    vs = vars_n(n)
    specs = list(expr_specs(n))
    rows = []

    if USE_RETURN:
        for suffix, expr, meta in specs:
            add_all(rows, f"return_{suffix}", n, vs, [f"return {render_expr(expr, vs, 'py')}"], lambda a, expr=expr: val_expr(expr, a), meta)

    for target in range(n):
        for suffix, expr, meta in specs:
            for name, mode in RULE_MODES:
                add_rule(rows, n, vs, f"{name}_{suffix}", target, expr, mode, meta)

    for k, family in ((2, "double_local_check"), (3, "triple_local_check"), (4, "quad_local_check")):
        if n >= k:
            for targets in combinations(range(n), k):
                for sources in product(range(n), repeat=k):
                    add_multi(rows, n, vs, family, targets, sources)

    rows = dedup(rows)
    rows.sort(key=lambda r: (r["n"], r["family"], r["text"], int(r["label"])))
    return rows


def build_data():
    train = puzzles("train")
    test = puzzles("test")
    train, val = split_train_val(train)

    ids = {
    "train": {p["uid"] for p in train},
    "val": {p["uid"] for p in val},
    "test": {p["uid"] for p in test},
    }

    overlaps_before = {
        "train_val": len(ids["train"] & ids["val"]),
        "train_test": len(ids["train"] & ids["test"]),
        "val_test": len(ids["val"] & ids["test"]),
    }
    print("[prepared] split UID overlaps before cleanup:", overlaps_before)

    test_ids = ids["test"]
    val_ids = ids["val"] - test_ids
    train_ids = ids["train"] - ids["val"] - ids["test"]

    skipped = {
        "train": len(ids["train"]) - len(train_ids),
        "val": len(ids["val"]) - len(val_ids),
        "test": 0,
    }
    print("[prepared] skipped overlapping UIDs:", skipped)

    train = [p for p in train if p["uid"] in train_ids]
    val = [p for p in val if p["uid"] in val_ids]
    test = [p for p in test if p["uid"] in test_ids]

    ids = {
        "train": {p["uid"] for p in train},
        "val": {p["uid"] for p in val},
        "test": {p["uid"] for p in test},
    }
    overlaps = {
        "train_val": len(ids["train"] & ids["val"]),
        "train_test": len(ids["train"] & ids["test"]),
        "val_test": len(ids["val"] & ids["test"]),
    }
    print("[prepared] split UID overlaps after cleanup:", overlaps)

    train_rows = train_phases(train)
    val_rows = full_rows(val, "val")
    test_rows = full_rows(test, "test")
    train_full = full_stats(train)

    train_paths = {name: os.path.join(PREP, f"english_train_{name}.jsonl") for name, *_ in PHASES}
    for name, path in train_paths.items():
        save_jsonl(path, project(train_rows[name]["rows"], "english_text"))

    paths = {
        "english_val": os.path.join(PREP, "english_val.jsonl"),
        "english_test": os.path.join(PREP, "english_test.jsonl"),
        "python_val": os.path.join(PREP, "python_val.jsonl"),
        "python_test": os.path.join(PREP, "python_test.jsonl"),
        "python_strict_atoms_train": os.path.join(PREP, "python_strict_atoms_train.jsonl"),
    }

    save_jsonl(paths["english_val"], project(val_rows, "english_text"))
    save_jsonl(paths["english_test"], project(test_rows, "english_text"))
    save_jsonl(paths["python_val"], project(val_rows, "python_text"))
    save_jsonl(paths["python_test"], project(test_rows, "python_text"))

    atom_rows = [r for n in NS for r in atom_rows_n(n)]
    save_jsonl(paths["python_strict_atoms_train"], atom_rows)

    cov = {name: cov_summary(train_rows[name]["coverage"]) for name in train_rows}
    save_json(os.path.join(PREP_RES, "english_phase_coverage_summary.json"), cov)
    save_json(os.path.join(PREP_RES, "english_phase_coverage_raw.json"), {name: train_rows[name]["coverage"] for name in train_rows})

    overview = {
        "dataset_id": DATASET_ID,
        "target_ns": list(NS),
        "split_protection": {
            "unit": "puzzle uid",
            "train_puzzles": len(ids["train"]),
            "val_puzzles": len(ids["val"]),
            "test_puzzles": len(ids["test"]),
            "overlaps": overlaps,
        },
        "counts": {
            "train_examples_full": train_full["num_examples"],
            "english_train_main": len(train_rows["main"]["rows"]),
            "english_train_finish": len(train_rows["finish"]["rows"]),
            "english_val": len(val_rows),
            "english_test": len(test_rows),
            "python_val": len(val_rows),
            "python_test": len(test_rows),
            "python_strict_atoms_train": len(atom_rows),
        },
        "positive_rates": {
            "train_full": train_full["positive_rate"],
            "english_train_main": rate(train_rows["main"]["rows"]),
            "english_train_finish": rate(train_rows["finish"]["rows"]),
            "english_val": rate(val_rows),
            "english_test": rate(test_rows),
            "python_strict_atoms_train": rate(atom_rows),
        },
        "artifacts": {
            "english_train_main": train_paths["main"],
            "english_train_finish": train_paths["finish"],
            **paths,
            "coverage_summary": os.path.join(PREP_RES, "english_phase_coverage_summary.json"),
            "coverage_raw": os.path.join(PREP_RES, "english_phase_coverage_raw.json"),
        },
    }

    save_json(os.path.join(PREP_RES, "prepared_data_overview.json"), overview)
    print("\nSaved prepared data to:")
    print(PREP)
    print("\nOverview:")
    print(json.dumps(overview, indent=2, ensure_ascii=False))


build_data()


In [ ]:
!pip -q install -U "transformers>=4.46.0" "peft>=0.13.0" "accelerate>=1.0.0" "torchao>=0.16.0" safetensors tqdm

import gc
import torch
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, get_linear_schedule_with_warmup

os.environ["TOKENIZERS_PARALLELISM"] = "false"

MODEL = "EleutherAI/pythia-1.4b-deduped"

ATOM_DIR = os.path.join(ROOT, "Python", "python_atoms_aux")
ATOM_RES = os.path.join(ATOM_DIR, "results")
ATOM_ALL = os.path.join(PREP, "python_strict_atoms_train.jsonl")
ATOM_TRAIN = os.path.join(ATOM_RES, "python_strict_atoms_train_split.jsonl")
ATOM_VAL = os.path.join(ATOM_RES, "python_strict_atoms_val_split.jsonl")
ATOM_BEST = os.path.join(ATOM_DIR, "best")
ATOM_LAST = os.path.join(ATOM_DIR, "last")

dirs(ATOM_RES, ATOM_BEST, ATOM_LAST)

SEED = 42
DEVICE = "cuda"
DTYPE = torch.float16

EPOCHS = 10
TRAIN_BS = 16
EVAL_BS = 16
GRAD_ACCUM = 1
LR = 5e-5
MAX_LEN = 256
WD = 0.01
WARMUP = 0.05
VAL_FRAC = 0.10

LORA_R = 64
LORA_ALPHA = 64
LORA_DROPOUT = 0.10
LORA_TARGETS = ["query_key_value", "dense", "dense_h_to_4h", "dense_4h_to_h"]

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


class ProbData(Dataset):
    def __init__(self, rows, tok):
        self.items = []
        for r in rows:
            x = tok(r["text"], add_special_tokens=False).input_ids[:MAX_LEN]
            self.items.append({
                "input_ids": x,
                "attention_mask": [1] * len(x),
                "labels": int(bool(r["label"])),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]


def batch(xs, pad):
    n = max(len(x["input_ids"]) for x in xs)
    return {
        "input_ids": torch.tensor([x["input_ids"] + [pad] * (n - len(x["input_ids"])) for x in xs]),
        "attention_mask": torch.tensor([x["attention_mask"] + [0] * (n - len(x["attention_mask"])) for x in xs]),
        "labels": torch.tensor([x["labels"] for x in xs]),
    }


def prob_loss(model, x, false_id, true_id):
    labels = x["labels"]
    y = model(input_ids=x["input_ids"], attention_mask=x["attention_mask"]).logits
    idx = x["attention_mask"].sum(dim=1) - 1
    z = y[torch.arange(y.shape[0], device=y.device), idx]
    pair = torch.stack([z[:, false_id], z[:, true_id]], dim=1)
    loss = torch.nn.functional.cross_entropy(pair.float(), labels)
    pred = pair.argmax(dim=1)
    return loss, pred


@torch.no_grad()
def eval_on(model, loader, false_id, true_id):
    model.eval()
    total = 0.0
    correct = 0
    count = 0
    for x in tqdm(loader, desc="atoms val", leave=False):
        x = {k: v.to(DEVICE) for k, v in x.items()}
        loss, pred = prob_loss(model, x, false_id, true_id)
        total += loss.item()
        correct += int((pred == x["labels"]).sum().item())
        count += int(x["labels"].numel())
    return total / max(1, len(loader)), correct / max(1, count)


tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token

neg_ids = tok(" False", add_special_tokens=False).input_ids
pos_ids = tok(" True", add_special_tokens=False).input_ids
print(f"[atoms] candidate token lengths | neg={len(neg_ids)} | pos={len(pos_ids)}")
NEG_ID = neg_ids[0]
POS_ID = pos_ids[0]

rows = load_jsonl(ATOM_ALL)
random.Random(SEED).shuffle(rows)

n_val = max(1, int(round(len(rows) * VAL_FRAC)))
val_rows = rows[:n_val]
train_rows = rows[n_val:]

overlap = {r["uid"] for r in train_rows} & {r["uid"] for r in val_rows}
print(f"[atoms] uid overlap train/val before cleanup: {len(overlap)}")

if overlap:
    train_rows = [r for r in train_rows if r["uid"] not in overlap]

overlap = {r["uid"] for r in train_rows} & {r["uid"] for r in val_rows}
print(f"[atoms] uid overlap train/val after cleanup: {len(overlap)}")

save_jsonl(ATOM_TRAIN, train_rows)
save_jsonl(ATOM_VAL, val_rows)

train_loader = DataLoader(
    ProbData(train_rows, tok),
    batch_size=TRAIN_BS,
    shuffle=True,
    collate_fn=lambda x: batch(x, tok.pad_token_id),
)
val_loader = DataLoader(
    ProbData(val_rows, tok),
    batch_size=EVAL_BS,
    shuffle=False,
    collate_fn=lambda x: batch(x, tok.pad_token_id),
)

print(f"[atoms] total rows: {len(rows)}")
print(f"[atoms] train rows: {len(train_rows)}")
print(f"[atoms] val rows:   {len(val_rows)}")
print(f"[atoms] uid overlap train/val: {len(overlap)}")

model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=DTYPE)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

cfg = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    inference_mode=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=LORA_TARGETS,
)

model = get_peft_model(model, cfg)
model.to(DEVICE)
model.print_trainable_parameters()

opt = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR, weight_decay=WD)
steps = max(1, math.ceil(len(train_loader) / GRAD_ACCUM) * EPOCHS)
sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(steps * WARMUP), num_training_steps=steps)

best = float("inf")
hist = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    opt.zero_grad(set_to_none=True)
    total = 0.0
    correct = 0
    count = 0
    pbar = tqdm(train_loader, desc=f"atoms epoch {epoch}/{EPOCHS}")

    for step, x in enumerate(pbar, start=1):
        x = {k: v.to(DEVICE) for k, v in x.items()}
        loss, pred = prob_loss(model, x, NEG_ID, POS_ID)
        (loss / GRAD_ACCUM).backward()
        total += loss.item()
        correct += int((pred == x["labels"]).sum().item())
        count += int(x["labels"].numel())

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)

        pbar.set_postfix(train_loss=f"{total / step:.4f}", train_acc=f"{correct / max(1, count):.4f}")

    train_loss = total / len(train_loader)
    train_acc = correct / max(1, count)
    val_loss, val_acc = eval_on(model, val_loader, NEG_ID, POS_ID)
    hist.append({"epoch": epoch, "train_loss": train_loss, "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc})
    print(f"[atoms] epoch {epoch} | train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | val_loss={val_loss:.4f} | val_acc={val_acc:.4f}")

    if val_loss < best:
        best = val_loss
        model.save_pretrained(ATOM_BEST, safe_serialization=True)
        tok.save_pretrained(ATOM_BEST)
        print(f"[atoms] saved best by val_loss: {ATOM_BEST}")

model.save_pretrained(ATOM_LAST, safe_serialization=True)
tok.save_pretrained(ATOM_LAST)

summary = {
    "model_id": MODEL,
    "objective": "cross_entropy_over_two_candidate_logits",
    "candidates": {"false": " False", "true": " True", "false_token_id": int(NEG_ID), "true_token_id": int(POS_ID)},
    "all_path": ATOM_ALL,
    "train_split_path": ATOM_TRAIN,
    "val_split_path": ATOM_VAL,
    "num_total_rows": len(rows),
    "num_train_rows": len(train_rows),
    "num_val_rows": len(val_rows),
    "uid_overlap_train_val": len(overlap),
    "epochs": EPOCHS,
    "train_bs": TRAIN_BS,
    "eval_bs": EVAL_BS,
    "grad_accum": GRAD_ACCUM,
    "lr": LR,
    "max_length": MAX_LEN,
    "weight_decay": WD,
    "warmup_ratio": WARMUP,
    "val_fraction": VAL_FRAC,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGETS,
    "best_val_loss": best,
    "adapter_best_dir": ATOM_BEST,
    "adapter_last_dir": ATOM_LAST,
    "selection_rule": "best adapter is selected by lowest validation two-candidate probability loss",
    "history": hist,
}

save_json(os.path.join(ATOM_RES, "python_atoms_train_summary.json"), summary)

del model
clean_gpu()

print("\nSaved atoms adapters:")
print("best:", ATOM_BEST)
print("last: ", ATOM_LAST)
print("summary:", os.path.join(ATOM_RES, "python_atoms_train_summary.json"))
